# Checkpoint 3: Data Transformation Pipeline
**Parametric Weather Insurance for US Crop Risk: Quantifying Basis Risk with Big Data on Databricks**  
Team No. 4, Big Data Project Work (ABDA Skill-Gap Capstone), XIME Bangalore

This notebook combines repository notebooks `01` to `08` (folder `databricks/`) in run order. It is written for **Databricks**: `spark` is the Databricks session, tables live in Unity Catalog (`workspace.default`) and raw files in Volumes. Import it with *Workspace > Import*, or run it from Jupyter through Databricks Connect. It will not run on a plain local Jupyter kernel.

Pipeline (medallion): **Bronze** (raw, 01 to 03) > **Silver** (cleaned, quality-filtered, linked, 04 to 07) > **Gold** (county-year weather features, 08).

Notebooks 01 to 03 download several GB (01: 9.47 GB of weather CSV). Skip them if the Bronze tables already exist.

In [ ]:
# Setup: on Databricks `spark` already exists. With Databricks Connect, uncomment the next two lines.
# from databricks.connect import DatabricksSession
# spark = DatabricksSession.builder.getOrCreate()
import json
try:
    spark
except NameError:
    raise RuntimeError('No Spark session found: run on Databricks or set up Databricks Connect')

---
## `01_ingest_noaa_ghcn`
```text
Ingest NOAA GHCN-Daily (2019-2025) from the public AWS Open Data bucket
into a Unity Catalog Volume, then load it into Delta tables.
Source: https://registry.opendata.aws/noaa-ghcn/  (no key needed)
```

In [ ]:
import json, os, shutil, requests
from concurrent.futures import ThreadPoolExecutor
from pyspark.sql import functions as F

CATALOG = spark.sql("SELECT current_catalog()").first()[0]
SCHEMA = "default"
YEARS = list(range(2019, 2026))
SRC = "https://noaa-ghcn-pds.s3.amazonaws.com"
HEADERS = {"User-Agent": "student-bigdata-project"}

spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.bigdata")
BASE = f"/Volumes/{CATALOG}/{SCHEMA}/bigdata/ghcn"
os.makedirs(f"{BASE}/by_year", exist_ok=True)
print(CATALOG, BASE)

In [ ]:
def download(url, dest):
    # Skip files already fully downloaded, so the notebook can be re-run safely.
    remote_size = int(requests.head(url, timeout=60, headers=HEADERS).headers["Content-Length"])
    if os.path.exists(dest) and os.path.getsize(dest) == remote_size:
        return dest, remote_size, "cached"
    with requests.get(url, stream=True, timeout=600, headers=HEADERS) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            shutil.copyfileobj(r.raw, f, length=16 * 1024 * 1024)
    assert os.path.getsize(dest) == remote_size, f"size mismatch for {url}"
    return dest, remote_size, "downloaded"

pairs = [(f"{SRC}/csv/by_year/{y}.csv", f"{BASE}/by_year/{y}.csv") for y in YEARS]
pairs.append((f"{SRC}/ghcnd-stations.txt", f"{BASE}/ghcnd-stations.txt"))

with ThreadPoolExecutor(4) as ex:
    downloads = list(ex.map(lambda p: download(*p), pairs))

raw_gb = sum(size for _, size, _ in downloads) / 1e9
for path, size, status in downloads:
    print(f"{os.path.basename(path):24s} {size/1e9:6.2f} GB  {status}")
print(f"Raw total: {raw_gb:.2f} GB")

In [ ]:
# Daily observations -> Delta. DATA_VALUE units depend on ELEMENT
# (e.g. TMAX/TMIN in tenths of degC, PRCP in tenths of mm).
schema = ("ID STRING, DATE STRING, ELEMENT STRING, DATA_VALUE INT, "
          "M_FLAG STRING, Q_FLAG STRING, S_FLAG STRING, OBS_TIME STRING")

daily = (spark.read.csv(f"{BASE}/by_year/*.csv", header=True, schema=schema)
    .withColumn("date", F.to_date("DATE", "yyyyMMdd"))
    .withColumn("year", F.year("date"))
    .withColumn("country_code", F.substring("ID", 1, 2))
    .select(F.col("ID").alias("station_id"), "date", "year", "country_code",
            F.col("ELEMENT").alias("element"), F.col("DATA_VALUE").alias("value"),
            F.col("M_FLAG").alias("m_flag"), F.col("Q_FLAG").alias("q_flag"),
            F.col("S_FLAG").alias("s_flag"), F.col("OBS_TIME").alias("obs_time")))

(daily.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("year")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.ghcn_daily"))

In [ ]:
# Station metadata is fixed-width (see GHCN-Daily readme section IV).
stations = (spark.read.text(f"{BASE}/ghcnd-stations.txt")
    .select(
        F.trim(F.substring("value", 1, 11)).alias("station_id"),
        F.trim(F.substring("value", 13, 8)).cast("double").alias("latitude"),
        F.trim(F.substring("value", 22, 9)).cast("double").alias("longitude"),
        F.trim(F.substring("value", 32, 6)).cast("double").alias("elevation_m"),
        F.trim(F.substring("value", 39, 2)).alias("state"),
        F.trim(F.substring("value", 42, 30)).alias("name"),
        F.substring("value", 1, 2).alias("country_code")))

(stations.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.ghcn_stations"))

In [ ]:
t = spark.table(f"{CATALOG}.{SCHEMA}.ghcn_daily")
per_year = {r["year"]: r["count"] for r in t.groupBy("year").count().collect()}
detail = spark.sql(f"DESCRIBE DETAIL {CATALOG}.{SCHEMA}.ghcn_daily").first()

summary = {
    "raw_csv_gb": round(raw_gb, 2),
    "daily_rows": sum(per_year.values()),
    "rows_per_year": dict(sorted(per_year.items())),
    "delta_gb": round(detail["sizeInBytes"] / 1e9, 2),
    "delta_files": detail["numFiles"],
    "distinct_stations": t.select("station_id").distinct().count(),
    "distinct_elements": t.select("element").distinct().count(),
    "station_rows": spark.table(f"{CATALOG}.{SCHEMA}.ghcn_stations").count(),
    "tables": [f"{CATALOG}.{SCHEMA}.ghcn_daily", f"{CATALOG}.{SCHEMA}.ghcn_stations"],
}
print(json.dumps(summary, indent=2))
print(json.dumps(summary))

---
## `02_ingest_nass_yields`
```text
Ingest USDA NASS Quick Stats crop data (county yields) and US Census county
centroids into a separate Unity Catalog Volume, then load them into Delta tables.
Sources (no key needed):
  https://www.nass.usda.gov/datasets/              (qs.crops_YYYYMMDD.txt.gz, refreshed daily)
  https://www.census.gov/geographies/reference-files/time-series/geo/gazetteer-files.html
```

In [ ]:
import json, os, re, shutil, zipfile, requests
from pyspark.sql import functions as F

CATALOG = spark.sql("SELECT current_catalog()").first()[0]
SCHEMA = "default"
HEADERS = {"User-Agent": "student-bigdata-project"}

spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.agri_yield")
BASE = f"/Volumes/{CATALOG}/{SCHEMA}/agri_yield"
os.makedirs(f"{BASE}/nass", exist_ok=True)
os.makedirs(f"{BASE}/census", exist_ok=True)

# The crops file name carries the publish date, so find the current one.
listing = requests.get("https://www.nass.usda.gov/datasets/", headers=HEADERS, timeout=60).text
CROPS_FILE = sorted(set(re.findall(r"qs\.crops_\d{8}\.txt\.gz", listing)))[-1]
NASS_URL = f"https://www.nass.usda.gov/datasets/{CROPS_FILE}"
GAZ_URL = "https://www2.census.gov/geo/docs/maps-data/data/gazetteer/2024_Gazetteer/2024_Gaz_counties_national.zip"
print(CATALOG, BASE, CROPS_FILE)

In [ ]:
def download(url, dest):
    # Skip files already fully downloaded, so the notebook can be re-run safely.
    remote_size = int(requests.head(url, timeout=60, headers=HEADERS, allow_redirects=True).headers["Content-Length"])
    if os.path.exists(dest) and os.path.getsize(dest) == remote_size:
        return remote_size, "cached"
    with requests.get(url, stream=True, timeout=1200, headers=HEADERS) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            shutil.copyfileobj(r.raw, f, length=16 * 1024 * 1024)
    assert os.path.getsize(dest) == remote_size, f"size mismatch for {url}"
    return remote_size, "downloaded"

nass_path = f"{BASE}/nass/{CROPS_FILE}"
gaz_zip = f"{BASE}/census/2024_Gaz_counties_national.zip"
nass_size, nass_status = download(NASS_URL, nass_path)
gaz_size, gaz_status = download(GAZ_URL, gaz_zip)
with zipfile.ZipFile(gaz_zip) as z:
    z.extractall(f"{BASE}/census")
print(f"NASS crops: {nass_size/1e9:.2f} GB compressed ({nass_status}); gazetteer: {gaz_size/1e3:.0f} KB ({gaz_status})")

In [ ]:
# Bronze: the full crops file as published (all strings). VALUE keeps NASS
# markers such as "(D)" = withheld to avoid disclosing individual farms.
nass_raw = spark.read.csv(nass_path, sep="\t", header=True)
nass_raw = nass_raw.toDF(*[re.sub(r"[^a-z0-9]+", "_", c.lower()).strip("_") for c in nass_raw.columns])

(nass_raw.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.nass_crops_raw"))

In [ ]:
# Silver: annual county-level yields, survey data only, numeric values.
nass = spark.table(f"{CATALOG}.{SCHEMA}.nass_crops_raw")
county_yields = (nass
    .filter((F.col("agg_level_desc") == "COUNTY") & (F.col("statisticcat_desc") == "YIELD")
            & (F.col("source_desc") == "SURVEY") & (F.col("freq_desc") == "ANNUAL")
            & (F.col("reference_period_desc") == "YEAR") & (F.col("county_ansi") != ""))
    # Serverless runs in ANSI mode, so try_cast turns "(D)" and similar markers into NULL.
    .withColumn("yield_value", F.expr("try_cast(regexp_replace(trim(value), ',', '') AS DOUBLE)"))
    .filter(F.col("yield_value").isNotNull())
    .select(
        F.concat(F.col("state_fips_code"), F.col("county_ansi")).alias("county_fips"),
        "state_alpha", "county_name",
        F.col("commodity_desc").alias("commodity"),
        F.col("short_desc").alias("series"),
        F.col("unit_desc").alias("unit"),
        F.expr("try_cast(year AS INT)").alias("year"),
        "yield_value"))

(county_yields.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.nass_county_yields"))

In [ ]:
# County centroids, used to link each county to nearby weather stations.
gaz_txt = [f for f in os.listdir(f"{BASE}/census") if f.endswith(".txt")][0]
gaz = spark.read.csv(f"{BASE}/census/{gaz_txt}", sep="\t", header=True)
gaz = gaz.toDF(*[c.strip().lower() for c in gaz.columns])
counties = gaz.select(
    F.col("geoid").alias("county_fips"),
    F.col("usps").alias("state_alpha"),
    F.col("name").alias("county_name"),
    F.expr("try_cast(trim(intptlat) AS DOUBLE)").alias("latitude"),
    F.expr("try_cast(trim(intptlong) AS DOUBLE)").alias("longitude"),
    F.expr("try_cast(trim(aland_sqmi) AS DOUBLE)").alias("land_sqmi"))

(counties.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.us_counties"))

In [ ]:
cy = spark.table(f"{CATALOG}.{SCHEMA}.nass_county_yields")
recent = cy.filter(F.col("year").between(2019, 2025))
top_series = (recent.groupBy("series").agg(F.countDistinct("county_fips").alias("counties"),
                                          F.count("*").alias("rows"))
              .orderBy(F.desc("rows")).limit(10).collect())
matched = (recent.select("county_fips").distinct()
           .join(spark.table(f"{CATALOG}.{SCHEMA}.us_counties"), "county_fips").count())

summary = {
    "crops_file": CROPS_FILE,
    "nass_compressed_gb": round(nass_size / 1e9, 2),
    "nass_raw_rows": spark.table(f"{CATALOG}.{SCHEMA}.nass_crops_raw").count(),
    "county_yield_rows_all_years": cy.count(),
    "county_yield_rows_2019_2025": recent.count(),
    "counties_with_yields_2019_2025": recent.select("county_fips").distinct().count(),
    "those_counties_matched_to_centroids": matched,
    "rows_per_year_2019_2025": {r["year"]: r["count"] for r in recent.groupBy("year").count().orderBy("year").collect()},
    "top_series_2019_2025": [{"series": r["series"], "counties": r["counties"], "rows": r["rows"]} for r in top_series],
    "us_counties": spark.table(f"{CATALOG}.{SCHEMA}.us_counties").count(),
    "tables": [f"{CATALOG}.{SCHEMA}.nass_crops_raw", f"{CATALOG}.{SCHEMA}.nass_county_yields",
               f"{CATALOG}.{SCHEMA}.us_counties"],
}
print(json.dumps(summary, indent=2))
print(json.dumps(summary))

---
## `03_extend_ghcn_history`
```text
Story 1.3: extend GHCN-Daily back to 2000 so each county has ~26 seasons.

Inputs : s3://noaa-ghcn-pds/csv/by_year/{2000..2018}.csv (public)
Output : appends years 2000-2018 to workspace.default.ghcn_daily (same schema, partitioned by year)
Assumptions (document any change here):
  - Only US stations (ID prefix "US") and elements PRCP, TMAX, TMIN are kept for the new years.
    This keeps storage small on the Free Edition and is all the trigger features need.
    Years 2019-2025 already in the table are unfiltered; downstream notebooks must filter the same way.
  - Each year is loaded on its own with replaceWhere, so the notebook is safe to re-run.
  - Each raw CSV is deleted after loading to save Volume space.
```

In [ ]:
import os, shutil, json, requests
from pyspark.sql import functions as F

CATALOG = spark.sql("SELECT current_catalog()").first()[0]
SCHEMA = "default"
TABLE = f"{CATALOG}.{SCHEMA}.ghcn_daily"
NEW_YEARS = list(range(2000, 2019))
KEEP_ELEMENTS = ["PRCP", "TMAX", "TMIN"]
SRC = "https://noaa-ghcn-pds.s3.amazonaws.com/csv/by_year"
HEADERS = {"User-Agent": "student-bigdata-project"}
BASE = f"/Volumes/{CATALOG}/{SCHEMA}/bigdata/ghcn/by_year"
os.makedirs(BASE, exist_ok=True)

SCHEMA_DDL = ("ID STRING, DATE STRING, ELEMENT STRING, DATA_VALUE INT, "
              "M_FLAG STRING, Q_FLAG STRING, S_FLAG STRING, OBS_TIME STRING")

In [ ]:
def already_loaded(year):
    return spark.table(TABLE).where(F.col("year") == year).limit(1).count() > 0

def load_year(year):
    dest = f"{BASE}/{year}.csv"
    with requests.get(f"{SRC}/{year}.csv", stream=True, timeout=600, headers=HEADERS) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            shutil.copyfileobj(r.raw, f, length=16 * 1024 * 1024)
    df = (spark.read.csv(dest, header=True, schema=SCHEMA_DDL)
        .where(F.substring("ID", 1, 2) == "US")
        .where(F.col("ELEMENT").isin(KEEP_ELEMENTS))
        .withColumn("date", F.to_date("DATE", "yyyyMMdd"))
        .select(F.col("ID").alias("station_id"), "date", F.year("date").alias("year"),
                F.substring("ID", 1, 2).alias("country_code"),
                F.col("ELEMENT").alias("element"), F.col("DATA_VALUE").alias("value"),
                F.col("M_FLAG").alias("m_flag"), F.col("Q_FLAG").alias("q_flag"),
                F.col("S_FLAG").alias("s_flag"), F.col("OBS_TIME").alias("obs_time")))
    (df.write.format("delta").mode("overwrite")
        .option("replaceWhere", f"year = {year}")
        .partitionBy("year").saveAsTable(TABLE))
    os.remove(dest)

In [ ]:
for y in NEW_YEARS:
    if already_loaded(y):
        print(y, "already loaded, skipping")
        continue
    load_year(y)
    print(y, "loaded")

In [ ]:
# Per-year counts (acceptance criterion: report per-year counts) and a second-method check via SQL.
per_year = spark.sql(f"""
    SELECT year, COUNT(*) AS rows, COUNT(DISTINCT station_id) AS stations
    FROM {TABLE} WHERE country_code = 'US' AND element IN ('PRCP','TMAX','TMIN')
    GROUP BY year ORDER BY year""")
per_year.show(40)

total_df = spark.table(TABLE).where("country_code = 'US' AND element IN ('PRCP','TMAX','TMIN')").count()
total_sql = sum(r["rows"] for r in per_year.collect())
assert total_df == total_sql, (total_df, total_sql)
missing = set(NEW_YEARS) - {r["year"] for r in per_year.collect()}
assert not missing, f"years missing: {missing}"
print(json.dumps({"us_rows_2000_2025": total_df}))

---
## `04_data_quality_report`
```text
Story 1.4: data-quality report.

Inputs : ghcn_daily, ghcn_stations, nass_county_yields, nass_crops_raw, us_counties
Output : workspace.default.dq_report (metric, scope, value) - one row per metric
Assumptions:
  - Weather metrics cover US stations and PRCP/TMAX/TMIN only (what the project uses).
  - A row "fails QC" when q_flag is not null.
  - Yield suppression share = non-numeric yield values such as "(D)" among county-level annual survey rows.
```

In [ ]:
from pyspark.sql import functions as F

CATALOG = spark.sql("SELECT current_catalog()").first()[0]
S = f"{CATALOG}.default"
rows = []  # (metric, scope, value)

w = (spark.table(f"{S}.ghcn_daily")
     .where("country_code = 'US' AND element IN ('PRCP','TMAX','TMIN')"))

In [ ]:
# 1. Null rate and QC-flag share, by element.
by_el = w.groupBy("element").agg(
    F.count("*").alias("n"),
    F.avg(F.col("value").isNull().cast("int")).alias("null_rate"),
    F.avg(F.col("q_flag").isNotNull().cast("int")).alias("qc_fail_share"))
for r in by_el.collect():
    rows += [("null_rate", r["element"], r["null_rate"]),
             ("qc_fail_share", r["element"], r["qc_fail_share"]),
             ("rows", r["element"], float(r["n"]))]

# 2. QC-flag share by year (does quality drift over time?).
for r in w.groupBy("year").agg(F.avg(F.col("q_flag").isNotNull().cast("int")).alias("s")).collect():
    rows.append(("qc_fail_share_by_year", str(r["year"]), r["s"]))

In [ ]:
# 3. Station coverage by state: stations with any PRCP and TMAX, per state, per year.
st = spark.table(f"{S}.ghcn_stations").select("station_id", "state")
cov = (w.where("q_flag IS NULL AND value IS NOT NULL")
       .groupBy("station_id", "year").agg(
           F.max((F.col("element") == "PRCP").cast("int")).alias("has_prcp"),
           F.max((F.col("element") == "TMAX").cast("int")).alias("has_tmax"))
       .join(st, "station_id")
       .groupBy("state", "year").agg(
           F.sum("has_prcp").alias("prcp_stations"),
           F.sum("has_tmax").alias("tmax_stations")))
for r in cov.collect():
    rows += [("stations_with_prcp", f"{r['state']}|{r['year']}", float(r["prcp_stations"])),
             ("stations_with_tmax", f"{r['state']}|{r['year']}", float(r["tmax_stations"]))]

In [ ]:
# 4. Yield suppression share. Raw file: county rows, annual survey, yield series.
raw = spark.table(f"{S}.nass_crops_raw")
cols = {c.lower(): c for c in raw.columns}
base = (raw.where(F.col(cols["agg_level_desc"]) == "COUNTY")
           .where(F.col(cols["source_desc"]) == "SURVEY")
           .where(F.col(cols["freq_desc"]) == "ANNUAL")
           .where(F.col(cols["statisticcat_desc"]) == "YIELD"))
tot = base.count()
numeric = base.where(F.expr(f"try_cast(regexp_replace(trim({cols['value']}), ',', '') AS DOUBLE) IS NOT NULL")).count()
rows += [("yield_rows_total", "county_annual_survey", float(tot)),
         ("yield_rows_numeric", "county_annual_survey", float(numeric)),
         ("yield_suppressed_share", "county_annual_survey", 1 - numeric / tot)]

# 5. Centroid match share (acceptance from story 1.2, re-checked).
y = spark.table(f"{S}.nass_county_yields").select("county_fips").distinct()
c = spark.table(f"{S}.us_counties").select("county_fips")
rows.append(("yield_county_centroid_match", "all",
             y.join(c, "county_fips").count() / y.count()))

In [ ]:
dq = spark.createDataFrame(rows, "metric STRING, scope STRING, value DOUBLE")
dq.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{S}.dq_report")
display(dq.where("metric IN ('null_rate','qc_fail_share','yield_suppressed_share','yield_county_centroid_match')"))
print("dq_report rows:", dq.count())

---
## `05_county_nearest_stations`
```text
Story 2.1: link each county to its k nearest GHCN stations.

Inputs : us_counties (centroids), ghcn_stations, ghcn_daily
Output : workspace.default.county_station_links (county_fips, station_id, rank, distance_km)
Assumptions:
  - Only counties that have yield data are linked (the project scope).
  - Only US stations that reported PRCP or TMAX in at least one year are candidates.
    Story 2.2 will apply the stricter >=90% growing-season completeness filter.
  - K = 10 nearest stations per county, within MAX_KM = 100 km (wide on purpose; 2.3 reports the 30 km gaps).
  - Distance is haversine from the county's internal-point centroid, in km.
```

In [ ]:
from pyspark.sql import functions as F, Window

CATALOG = spark.sql("SELECT current_catalog()").first()[0]
S = f"{CATALOG}.default"
K, MAX_KM = 10, 100.0

counties = (spark.table(f"{S}.us_counties")
    .join(spark.table(f"{S}.nass_county_yields").select("county_fips").distinct(), "county_fips")
    .select("county_fips", F.col("latitude").alias("clat"), F.col("longitude").alias("clon")))

active = (spark.table(f"{S}.ghcn_daily")
    .where("country_code = 'US' AND element IN ('PRCP','TMAX') AND q_flag IS NULL")
    .select("station_id").distinct())
stations = (spark.table(f"{S}.ghcn_stations").join(active, "station_id")
    .select("station_id", F.col("latitude").alias("slat"), F.col("longitude").alias("slon")))
print("counties:", counties.count(), "candidate stations:", stations.count())

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = (F.col(c) for c in (lat1, lon1, lat2, lon2))
    p1, p2 = F.radians(lat1), F.radians(lat2)
    dphi, dlam = F.radians(lat2 - lat1), F.radians(lon2 - lon1)
    a = F.sin(dphi / 2) ** 2 + F.cos(p1) * F.cos(p2) * F.sin(dlam / 2) ** 2
    return 2 * 6371.0088 * F.asin(F.sqrt(a))

# Cheap bounding-box prefilter (1 degree latitude ~ 111 km) avoids a full 3k x 10k+ cross join cost
# on wide boxes; stations is small enough to broadcast.
pad = MAX_KM / 111.0
joined = (counties.join(F.broadcast(stations),
        (F.abs(F.col("clat") - F.col("slat")) <= pad) &
        (F.abs(F.col("clon") - F.col("slon")) <= pad / F.cos(F.radians(F.col("clat")))))
    .withColumn("distance_km", haversine_km("clat", "clon", "slat", "slon"))
    .where(F.col("distance_km") <= MAX_KM))

w = Window.partitionBy("county_fips").orderBy("distance_km", "station_id")
links = (joined.withColumn("rank", F.row_number().over(w))
    .where(F.col("rank") <= K)
    .select("county_fips", "station_id", "rank", F.round("distance_km", 3).alias("distance_km")))

(links.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{S}.county_station_links"))

In [ ]:
# Checks: second method (pure SQL, one sample county) and coverage summary.
t = spark.table(f"{S}.county_station_links")
print("link rows:", t.count(), "counties linked:", t.select("county_fips").distinct().count(),
      "of", counties.count())
t.where("rank = 1").selectExpr(
    "percentile_approx(distance_km, array(0.5, 0.9, 0.99)) AS nearest_km_p50_p90_p99",
    "max(distance_km) AS max_nearest_km").show(truncate=False)

sample = t.where("rank = 1").orderBy("county_fips").first()
check = spark.sql(f"""
    SELECT s.station_id,
      2*6371.0088*asin(sqrt(pow(sin(radians(s.latitude-c.latitude)/2),2) +
        cos(radians(c.latitude))*cos(radians(s.latitude))*pow(sin(radians(s.longitude-c.longitude)/2),2))) AS d
    FROM {S}.ghcn_stations s JOIN {S}.us_counties c ON c.county_fips = '{sample['county_fips']}'
    WHERE s.station_id IN (SELECT station_id FROM {S}.county_station_links WHERE county_fips = '{sample['county_fips']}')
    ORDER BY d LIMIT 1""").first()
assert check["station_id"] == sample["station_id"] and abs(check["d"] - sample["distance_km"]) < 0.01
print("independent SQL check passed for county", sample["county_fips"])

---
## `06_usable_stations`
```text
Story 2.2: keep only station-years with usable growing-season data.

Inputs : ghcn_daily (US; PRCP, TMAX)
Output : workspace.default.station_year_quality (station_id, year, prcp_days, tmax_days,
         prcp_completeness, tmax_completeness, usable)
Assumptions:
  - Growing season = 1 April to 30 September = 183 days.
  - A day counts only if the value is not null and q_flag is null (passed NOAA QC).
  - A station-year is usable when BOTH PRCP and TMAX completeness are >= MIN_COMPLETENESS.
  - Usability is judged per station-year, not per station, because stations open and close.
```

In [ ]:
from pyspark.sql import functions as F

CATALOG = spark.sql("SELECT current_catalog()").first()[0]
S = f"{CATALOG}.default"
MIN_COMPLETENESS = 0.90
SEASON_DAYS = 183

d = (spark.table(f"{S}.ghcn_daily")
     .where("country_code = 'US' AND element IN ('PRCP','TMAX') AND q_flag IS NULL AND value IS NOT NULL")
     .where(F.month("date").between(4, 9)))

q = (d.groupBy("station_id", "year").agg(
        F.countDistinct(F.when(F.col("element") == "PRCP", F.col("date"))).alias("prcp_days"),
        F.countDistinct(F.when(F.col("element") == "TMAX", F.col("date"))).alias("tmax_days"))
     .withColumn("prcp_completeness", F.round(F.col("prcp_days") / SEASON_DAYS, 4))
     .withColumn("tmax_completeness", F.round(F.col("tmax_days") / SEASON_DAYS, 4))
     .withColumn("usable", (F.col("prcp_completeness") >= MIN_COMPLETENESS) &
                           (F.col("tmax_completeness") >= MIN_COMPLETENESS)))

(q.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{S}.station_year_quality"))

In [ ]:
t = spark.table(f"{S}.station_year_quality")
t.groupBy("year").agg(F.count("*").alias("station_years"),
                      F.sum(F.col("usable").cast("int")).alias("usable")).orderBy("year").show(30)
assert t.where("prcp_days > 183 OR tmax_days > 183").count() == 0, "more days than the season has"

---
## `07_county_coverage`
```text
Story 2.3: which counties lack a usable station nearby?

Inputs : county_station_links (10 nearest stations within 100 km), station_year_quality, nass_county_yields
Output : workspace.default.county_year_station (county_fips, year, station_id, distance_km)
           = the NEAREST USABLE station per county-year, any distance up to 100 km
         workspace.default.county_coverage_gaps (county_fips, state_fips, years_total, years_covered_30km,
           nearest_usable_km_median)
Assumptions:
  - "Usable" comes from station_year_quality (>=90% growing-season PRCP and TMAX).
  - Covered = nearest usable station is within COVER_KM = 30 km in that year.
  - Scope = counties that have yield data. Corn Belt = state FIPS IA 19, IL 17, IN 18, NE 31, MN 27.
  - Links hold only the 10 nearest stations, so a county whose 10 nearest are all unusable counts as uncovered.
```

In [ ]:
from pyspark.sql import functions as F, Window

CATALOG = spark.sql("SELECT current_catalog()").first()[0]
S = f"{CATALOG}.default"
COVER_KM = 30.0
YEARS = list(range(2000, 2026))
CORN_BELT = ["19", "17", "18", "31", "27"]

links = spark.table(f"{S}.county_station_links")
usable = spark.table(f"{S}.station_year_quality").where("usable").select("station_id", "year")
years = spark.createDataFrame([(y,) for y in YEARS], "year INT")

cy = (links.crossJoin(years).join(usable, ["station_id", "year"]))
w = Window.partitionBy("county_fips", "year").orderBy("distance_km", "station_id")
nearest = (cy.withColumn("r", F.row_number().over(w)).where("r = 1")
           .select("county_fips", "year", "station_id", "distance_km"))
(nearest.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{S}.county_year_station"))

In [ ]:
scope = (spark.table(f"{S}.nass_county_yields").select("county_fips").distinct()
         .crossJoin(years))
cov = (scope.join(spark.table(f"{S}.county_year_station"), ["county_fips", "year"], "left")
       .withColumn("covered", (F.col("distance_km") <= COVER_KM).cast("int")))
gaps = (cov.groupBy("county_fips").agg(
            F.count("*").alias("years_total"),
            F.sum(F.coalesce("covered", F.lit(0))).alias("years_covered_30km"),
            F.percentile_approx("distance_km", 0.5).alias("nearest_usable_km_median"))
        .withColumn("state_fips", F.substring("county_fips", 1, 2)))
(gaps.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{S}.county_coverage_gaps"))

In [ ]:
g = spark.table(f"{S}.county_coverage_gaps")
print("all counties with yields:", g.count())
g.selectExpr("sum(case when years_covered_30km = years_total then 1 else 0 end) AS always_covered",
             "sum(case when years_covered_30km = 0 then 1 else 0 end) AS never_covered").show()
cb = g.where(F.col("state_fips").isin(CORN_BELT))
cb.selectExpr("count(*) AS corn_belt_counties",
              "sum(case when years_covered_30km = years_total then 1 else 0 end) AS always_covered",
              "sum(case when years_covered_30km = 0 then 1 else 0 end) AS never_covered",
              "round(avg(years_covered_30km / years_total), 3) AS avg_share_years_covered").show()

---
## `08_weather_features`
```text
Stories 3.1 and 3.2: weather index features per county-year (Gold).

Inputs : county_year_station (nearest usable station), ghcn_daily
Output : workspace.default.county_year_weather_features
  county_fips, year, station_id, distance_km,
  season_prcp_mm        April-September total precipitation (mm)
  min_30d_prcp_mm       driest 30-day window inside April-September (mm)
  days_tmax_gt30        July-August days with TMAX > 30 C
  days_tmax_gt35        July-August days with TMAX > 35 C
Assumptions:
  - One station per county-year: the nearest usable one, only if within MAX_KM = 30 km.
  - Raw units: PRCP in tenths of mm, TMAX in tenths of C. Converted here.
  - Days with a QC flag or null value are ignored, NOT treated as zero. A usable station-year has
    >=90% of days, so season_prcp_mm is a slight underestimate; missing days are not imputed.
  - 30-day window = 30 calendar days ending on a season day from 30 April on, needs >= 27 valid days.
```

In [ ]:
from pyspark.sql import functions as F, Window

CATALOG = spark.sql("SELECT current_catalog()").first()[0]
S = f"{CATALOG}.default"
MAX_KM = 30.0

pairs = (spark.table(f"{S}.county_year_station").where(F.col("distance_km") <= MAX_KM))
station_years = pairs.select("station_id", "year").distinct()

d = (spark.table(f"{S}.ghcn_daily")
     .where("country_code = 'US' AND element IN ('PRCP','TMAX') AND q_flag IS NULL AND value IS NOT NULL")
     .where(F.month("date").between(4, 9))
     .join(F.broadcast(station_years), ["station_id", "year"]))

In [ ]:
# Station-year features.
prcp = d.where("element = 'PRCP'").select("station_id", "year", "date", (F.col("value") / 10).alias("mm"))
season = prcp.groupBy("station_id", "year").agg(F.round(F.sum("mm"), 1).alias("season_prcp_mm"))

w30 = (Window.partitionBy("station_id", "year")
       .orderBy(F.col("date").cast("timestamp").cast("long") / 86400)
       .rangeBetween(-29, 0))
roll = (prcp.withColumn("sum30", F.sum("mm").over(w30)).withColumn("n30", F.count("mm").over(w30))
        .where((F.col("n30") >= 27) & (F.col("date") >= F.make_date(F.col("year"), F.lit(4), F.lit(30))))
        .groupBy("station_id", "year").agg(F.round(F.min("sum30"), 1).alias("min_30d_prcp_mm")))

heat = (d.where("element = 'TMAX' AND month(date) IN (7, 8)")
        .groupBy("station_id", "year").agg(
            F.sum((F.col("value") > 300).cast("int")).alias("days_tmax_gt30"),
            F.sum((F.col("value") > 350).cast("int")).alias("days_tmax_gt35")))

feat = (pairs.join(season, ["station_id", "year"], "left")
              .join(roll, ["station_id", "year"], "left")
              .join(heat, ["station_id", "year"], "left")
        .select("county_fips", "year", "station_id", "distance_km",
                "season_prcp_mm", "min_30d_prcp_mm", "days_tmax_gt30", "days_tmax_gt35"))

(feat.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{S}.county_year_weather_features"))

In [ ]:
t = spark.table(f"{S}.county_year_weather_features")
print("county-years:", t.count(), "counties:", t.select("county_fips").distinct().count())
t.selectExpr("round(avg(season_prcp_mm),0) AS avg_season_mm", "round(avg(min_30d_prcp_mm),1) AS avg_min30_mm",
             "round(avg(days_tmax_gt30),1) AS avg_gt30", "round(avg(days_tmax_gt35),1) AS avg_gt35",
             "sum(case when season_prcp_mm is null then 1 else 0 end) AS null_season",
             "sum(case when min_30d_prcp_mm is null then 1 else 0 end) AS null_min30").show()
assert t.where("days_tmax_gt35 > days_tmax_gt30 OR days_tmax_gt30 > 62").count() == 0, "heat counts impossible"
assert t.groupBy("county_fips", "year").count().where("count > 1").count() == 0, "duplicate county-year"